# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [1]:
# Install required packages (uncomment to run)
# !pip install -U langchain langchain-openai python-dotenv

In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [4]:
# YOUR TURN: Hawkins records tool

@tool
def check_hawkins_records(query: str) -> str:
    """Search the official Hawkins records about the Upside Down.
    Available topics: portal, monsters, psychics, electricity.
    Use this to find facts related to a complaint."""
    records = {
        "portal": "Record #001: A portal to the Upside Down opened under Hawkins Lab in 1983. Closed, then reopened. Status: unstable.",
        "monsters": "Record #002: Known creatures include the Demogorgon, Demodogs and the Mind Flayer. Demodogs travel in packs.",
        "psychics": "Record #003: Subject 'Eleven' shows telekinesis and can reach the Upside Down mentally. Nosebleeds after use.",
        "electricity": "Record #004: Flickering lights and power surges happen when creatures from the Upside Down are close by.",
    }
    for key in records:
        if key in query.lower():
            return records[key]
    return f"No record found in the Hawkins files for: '{query}'."

### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [5]:
# YOUR TURN: spell tool

@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Cast interdimensional spells to solve a problem creatively.
    creativity_level can be 'low', 'medium' or 'high' (more spells = more creative).
    Use this when the complaint asks for a creative solution."""
    levels = {"low": 1, "medium": 2, "high": 3}
    n = levels.get(creativity_level.lower(), 2)

    spells = [
        f"Spell of Christmas Lights: string lights around '{problem}' and ask it yes/no questions.",
        f"Eggo Ward: place a stack of waffles near '{problem}' to distract whatever is causing it.",
        f"Walkie-Talkie Banishment: shout 'Code Red!' at '{problem}' over channel 6.",
        f"Kate Bush Shield: play 'Running Up That Hill' at full volume until '{problem}' goes away.",
    ]

    chosen = random.sample(spells, n)
    return "\n".join(chosen)

### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [6]:
# YOUR TURN: party wisdom tool

@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the party (Mike, Dustin, Lucas, Will) for advice on a question or plan.
    Mention a party member's name to get their specific opinion."""
    party = {
        "mike": "Mike: 'We stick together. Nobody goes into the Upside Down alone. That's the plan.'",
        "dustin": "Dustin: 'Science says it's magnetic. Get a compass, follow where it points wrong.'",
        "lucas": "Lucas: 'Scout it first, fight second. Bring the wrist rocket.'",
        "will": "Will: 'If the back of your neck feels cold, it's close. Trust that feeling.'",
    }
    for name in party:
        if name in question.lower():
            return party[name]
    return "The party huddles in Mike's basement and agrees: stick together, grab the walkie-talkies, and don't trust the Demogorgon."

In [7]:
# YOUR TURN: test each tool on its own with .invoke({...})

print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon ignores 'My lights keep flickering' and walks back into the wall. Classic.
Record #001: A portal to the Upside Down opened under Hawkins Lab in 1983. Closed, then reopened. Status: unstable.
Eggo Ward: place a stack of waffles near 'Demodogs in my garden' to distract whatever is causing it.
Spell of Christmas Lights: string lights around 'Demodogs in my garden' and ask it yes/no questions.
Kate Bush Shield: play 'Running Up That Hill' at full volume until 'Demodogs in my garden' goes away.
The party huddles in Mike's basement and agrees: stick together, grab the walkie-talkies, and don't trust the Demogorgon.


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [ ]:
# YOUR TURN: tool list + system prompt + agent

tools = [consult_demogorgon, check_hawkins_records, cast_interdimensional_spell, gather_party_wisdom]

system_prompt = """You are the creative complaint handler for NormalObjects, a company dealing with strange events in Hawkins.

Rules:
1. Always call at least one tool before answering. Never answer without using a tool.
2. Be creative: combine several tools if they help (records for facts, party for advice, spells for solutions, Demogorgon for chaos).
3. Answer ONLY using information returned by the tools. Do not invent facts, records or names.
4. If a tool says "No record found", say clearly that no record exists. Do not make one up.
5. At the end of your answer, list the tools you used, like this:
   Tools used: tool_a, tool_b"""

agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [10]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# YOUR TURN: run every complaint, keep the results for Step 5

results = []
for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)
    print("COMPLAINT:", complaint)
    print("ANSWER:", result["messages"][-1].content)
    print("-" * 60)

COMPLAINT: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
ANSWER: It seems that your flickering Christmas lights and the strange noises from the wall might be related to creatures from the Upside Down. According to Hawkins records, such flickering and power surges typically occur when these creatures are nearby. 

As for the Demogorgon, it humorously ignored your complaint and just walked back into the wall, which is quite classic behavior for it!

Tools used: check_hawkins_records, consult_demogorgon
------------------------------------------------------------
COMPLAINT: There is a weird portal in my basement. What's the plan?
ANSWER: According to the official Hawkins records, there was a portal to the Upside Down that opened under Hawkins Lab in 1983. It was closed but has since reopened, and its status is currently unstable. 

The party's advice is to stick together, grab your walkie-talkies, and absolutely do not trust the Demogorgon. 

So, t

**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [11]:
# YOUR TURN: tool order + counts

from collections import Counter

all_tools = []

for i, r in enumerate(results):
    order = []
    for msg in r["messages"]:
        if msg.type == "ai" and msg.tool_calls:
            for tc in msg.tool_calls:
                order.append(tc["name"])
    all_tools.extend(order)
    print(f"Complaint {i + 1}: " + " → ".join(order))

print()
print("Total uses per tool:")
counts = Counter(all_tools)
for name, count in counts.items():
    print(f"  {name}: {count}")

Complaint 1: check_hawkins_records → consult_demogorgon
Complaint 2: check_hawkins_records → gather_party_wisdom
Complaint 3: cast_interdimensional_spell
Complaint 4: check_hawkins_records → consult_demogorgon → gather_party_wisdom

Total uses per tool:
  check_hawkins_records: 3
  consult_demogorgon: 2
  gather_party_wisdom: 2
  cast_interdimensional_spell: 1


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [12]:
# YOUR TURN: evidence trace for one result

for msg in results[0]["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print("TOOL CALL:", tc["name"])
            print("  ARGS:", tc["args"])
    elif msg.type == "tool":
        print("TOOL OUTPUT (" + msg.name + "):")
        print("  " + msg.content)
        print()

print("FINAL ANSWER:")
print(results[0]["messages"][-1].content)

TOOL CALL: check_hawkins_records
  ARGS: {'query': 'electricity'}
TOOL CALL: consult_demogorgon
  ARGS: {'complaint': 'Christmas lights flickering and noises from the wall.'}
TOOL OUTPUT (check_hawkins_records):
  Record #004: Flickering lights and power surges happen when creatures from the Upside Down are close by.

TOOL OUTPUT (consult_demogorgon):
  The Demogorgon ignores 'Christmas lights flickering and noises from the wall.' and walks back into the wall. Classic.

FINAL ANSWER:
It seems that your flickering Christmas lights and the strange noises from the wall might be related to creatures from the Upside Down. According to Hawkins records, such flickering and power surges typically occur when these creatures are nearby. 

As for the Demogorgon, it humorously ignored your complaint and just walked back into the wall, which is quite classic behavior for it!

Tools used: check_hawkins_records, consult_demogorgon


**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [13]:
# YOUR TURN: fixed-order handler

def structured_handler(complaint: str) -> str:
    record = check_hawkins_records.invoke({"query": complaint})
    wisdom = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke({"problem": complaint, "creativity_level": "low"})

    return (
        "RECORDS: " + record + "\n"
        + "PARTY: " + wisdom + "\n"
        + "SPELL: " + spell
    )

print(structured_handler(complaints[1]))

RECORDS: Record #001: A portal to the Upside Down opened under Hawkins Lab in 1983. Closed, then reopened. Status: unstable.
PARTY: The party huddles in Mike's basement and agrees: stick together, grab the walkie-talkies, and don't trust the Demogorgon.
SPELL: Spell of Christmas Lights: string lights around 'There is a weird portal in my basement. What's the plan?' and ask it yes/no questions.


| | Agent | Structured |
|---|---|---|
| Same tools every run? | | |
| Tool order predictable? | | |
| Can invent facts? | | |
| Easy to trace? | | |
| Best for which client case? | | |

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |